# MeatLens Folder Inference Notebook
## MobileNetV3Small Segmented6 Hybrid Best Model

This notebook loads the exported best segmented6 hybrid model and runs folder inference on new images.

It follows the segmented6 hybrid pipeline:
- center-square resize to 224x224
- HSV/LAB threshold segmentation
- segmented ROI background fill
- color and texture feature extraction
- saved feature fill values and saved scaler
- two-input hybrid prediction using image + handcrafted features

It does not start training.

In [1]:
import io
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_mobilenetv3

from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
    IPYWIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    IPYWIDGETS_AVAILABLE = False

import mobilenetv3small_segmented6_hybrid_lib as segmented6_lib
from apply_hsv_lab_threshold_roi_batch import process_image

print("TensorFlow:", tf.__version__)
print("ipywidgets available:", IPYWIDGETS_AVAILABLE)
print("JOBLIB_AVAILABLE in training library:", segmented6_lib.JOBLIB_AVAILABLE)
print("SKIMAGE_AVAILABLE in training library:", segmented6_lib.SKIMAGE_AVAILABLE)
print("CV2_AVAILABLE in training library:", segmented6_lib.CV2_AVAILABLE)

[INFO] TensorFlow XLA/JIT disabled.
TensorFlow: 2.10.0
ipywidgets available: True
JOBLIB_AVAILABLE in training library: True
SKIMAGE_AVAILABLE in training library: True
CV2_AVAILABLE in training library: True


In [2]:
PROJECT_ROOT = Path.cwd()
HYBRID_ROOT = PROJECT_ROOT / "training_outputs" / "mobilenetv3small_segmented6_hybrid"
MODELS_ROOT = HYBRID_ROOT / "models"
INFERENCE_ROOT = HYBRID_ROOT / "inference"
INFERENCE_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_ROOT / "meatlens_best_segmented6_hybrid_mobilenetv3small.h5"
METADATA_PATH = MODELS_ROOT / "meatlens_best_segmented6_hybrid_mobilenetv3small_metadata.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model not found: {MODEL_PATH}")
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Best model metadata not found: {METADATA_PATH}")

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

LABEL_ORDER = list(metadata.get("label_order", ["fresh", "not fresh", "spoiled"]))
FEATURE_COLUMNS = list(metadata.get("feature_columns", []))
INPUT_SIZE = tuple(metadata.get("input_size", [224, 224, 3]))
MODEL_INPUT_MODE = str(metadata.get("model_input_mode", ""))
IMAGE_CROP_MODE = str(metadata.get("image_crop_mode", ""))
SCALER_PATH = Path(metadata.get("scaler_path", ""))
FEATURE_FILL_VALUES_PATH = Path(metadata.get("feature_fill_values_path", ""))
BEST_FOLD = str(metadata.get("fold", ""))
BEST_SEED = int(metadata.get("seed", 0))
BACKGROUND_MODE = "gray"
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

if MODEL_INPUT_MODE != "cnn_plus_color_texture":
    raise ValueError(f"Unexpected model_input_mode: {MODEL_INPUT_MODE}")
if IMAGE_CROP_MODE != "preprocessed_segmented_roi_224":
    raise ValueError(f"Unexpected image_crop_mode: {IMAGE_CROP_MODE}")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Scaler not found: {SCALER_PATH}")
if not FEATURE_FILL_VALUES_PATH.exists():
    raise FileNotFoundError(f"Feature fill values CSV not found: {FEATURE_FILL_VALUES_PATH}")

print("MODEL_PATH:", MODEL_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("SCALER_PATH:", SCALER_PATH)
print("FEATURE_FILL_VALUES_PATH:", FEATURE_FILL_VALUES_PATH)
print("Best exported model came from fold:", BEST_FOLD)
print("Best exported model seed:", BEST_SEED)
print("MODEL_INPUT_MODE:", MODEL_INPUT_MODE)
print("IMAGE_CROP_MODE:", IMAGE_CROP_MODE)
print("INPUT_SIZE:", INPUT_SIZE)
print("LABEL_ORDER:", LABEL_ORDER)
print("FEATURE_COLUMNS:", len(FEATURE_COLUMNS))

MODEL_PATH: e:\Thesis Code\training_outputs\mobilenetv3small_segmented6_hybrid\models\meatlens_best_segmented6_hybrid_mobilenetv3small.h5
METADATA_PATH: e:\Thesis Code\training_outputs\mobilenetv3small_segmented6_hybrid\models\meatlens_best_segmented6_hybrid_mobilenetv3small_metadata.json
SCALER_PATH: e:\Thesis Code\training_outputs\mobilenetv3small_segmented6_hybrid\models\segmented6_hybrid_fold5_seed42_scaler.joblib
FEATURE_FILL_VALUES_PATH: e:\Thesis Code\training_outputs\mobilenetv3small_segmented6_hybrid\features\fold5_feature_fill_values.csv
Best exported model came from fold: fold5
Best exported model seed: 42
MODEL_INPUT_MODE: cnn_plus_color_texture
IMAGE_CROP_MODE: preprocessed_segmented_roi_224
INPUT_SIZE: (224, 224, 3)
LABEL_ORDER: ['fresh', 'not fresh', 'spoiled']
FEATURE_COLUMNS: 30


In [3]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
scaler = joblib.load(SCALER_PATH)
feature_fill_df = pd.read_csv(FEATURE_FILL_VALUES_PATH)
feature_fill_values = dict(zip(feature_fill_df["feature_name"].astype(str), feature_fill_df["fill_value"].astype(float)))

print("Model loaded successfully")
print("Scaler loaded successfully")
print("Feature fill values loaded:", len(feature_fill_values))
print("Model input names:", model.input_names if hasattr(model, "input_names") else "unavailable")

Model loaded successfully
Scaler loaded successfully
Feature fill values loaded: 30
Model input names: ['image_input', 'feature_input']


In [4]:
def list_images_recursive(folder_path):
    folder = Path(folder_path)
    if not folder.exists() or not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTS])


def normalize_uploaded_items(upload_value):
    items = []

    if isinstance(upload_value, dict):
        for name, payload in upload_value.items():
            content = payload.get("content") if isinstance(payload, dict) else None
            if content is not None:
                items.append({"name": name, "content": content})
        return items

    if isinstance(upload_value, (list, tuple)):
        for payload in upload_value:
            if isinstance(payload, dict):
                name = payload.get("name", "uploaded.zip")
                content = payload.get("content")
                if content is not None:
                    items.append({"name": name, "content": content})
        return items

    return items


def extract_uploaded_zip_to_temp(upload_item):
    temp_root = INFERENCE_ROOT / "uploaded_zips"
    temp_root.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    work_dir = temp_root / f"zip_{stamp}"
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    zip_path = work_dir / upload_item["name"]
    zip_path.write_bytes(upload_item["content"])

    extract_dir = work_dir / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(upload_item["content"]), "r") as zf:
        zf.extractall(extract_dir)

    return extract_dir

In [5]:
def build_feature_vector_from_segmented_roi(segmented_uint8):
    feature_dict = segmented6_lib.extract_color_texture_features_from_uint8(segmented_uint8)
    feature_df = pd.DataFrame([feature_dict])
    for col in FEATURE_COLUMNS:
        if col not in feature_df.columns:
            feature_df[col] = np.nan
    feature_df = feature_df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    feature_df = feature_df.fillna(pd.Series(feature_fill_values, dtype=np.float32))
    feature_df = feature_df.fillna(0.0)
    scaled = scaler.transform(feature_df)
    return scaled.astype(np.float32), feature_df.iloc[0].to_dict()


def save_processed_image(image_uint8, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(image_uint8).save(out_path, quality=95)
    return out_path


def predict_single_image(image_path, processed_output_path=None, background_mode=BACKGROUND_MODE):
    image_path = Path(image_path)
    segmented_uint8, seg_meta = process_image(image_path, background_mode=background_mode)

    if processed_output_path is not None:
        save_processed_image(segmented_uint8, processed_output_path)

    image_batch = np.expand_dims(preprocess_mobilenetv3(segmented_uint8.astype(np.float32).copy()), axis=0)
    feature_batch, raw_feature_dict = build_feature_vector_from_segmented_roi(segmented_uint8)

    probs = model.predict(
        {
            "image_input": image_batch.astype(np.float32),
            "feature_input": feature_batch.astype(np.float32),
        },
        verbose=0,
    )[0]

    sorted_idx = np.argsort(-probs)
    pred_idx = int(sorted_idx[0])
    second_idx = int(sorted_idx[1])

    pred_class = LABEL_ORDER[pred_idx]
    second_class = LABEL_ORDER[second_idx]
    top_confidence = float(probs[pred_idx])
    second_confidence = float(probs[second_idx])
    prediction_margin = float(top_confidence - second_confidence)
    top2_combined_confidence = float(top_confidence + second_confidence)

    prob_fresh = float(probs[segmented6_lib.LABEL_TO_INDEX["fresh"]])
    prob_not_fresh = float(probs[segmented6_lib.LABEL_TO_INDEX["not fresh"]])
    prob_spoiled = float(probs[segmented6_lib.LABEL_TO_INDEX["spoiled"]])

    freshness_score = segmented6_lib.compute_freshness_score(prob_fresh, prob_not_fresh, prob_spoiled)
    transition_label, recommendation = segmented6_lib.transition_label_and_recommendation(
        pred_class,
        top_confidence,
        second_class,
    )
    freshness_band = segmented6_lib.freshness_score_to_band(freshness_score)

    row = {
        "image_path": str(image_path),
        "file_name": image_path.name,
        "predicted_class": pred_class,
        "second_choice_class": second_class,
        "top_confidence": round(top_confidence, 6),
        "second_confidence": round(second_confidence, 6),
        "prediction_margin": round(prediction_margin, 6),
        "top2_combined_confidence": round(top2_combined_confidence, 6),
        "freshness_score": round(float(freshness_score), 2),
        "freshness_band": freshness_band,
        "transition_label": transition_label,
        "recommendation": recommendation,
        "segmentation_failed": bool(seg_meta.get("segmentation_failed", False)),
        "mask_area_ratio": round(float(seg_meta.get("mask_area_ratio", 0.0)), 6),
        "center_overlap_ratio": round(float(seg_meta.get("center_overlap_ratio", 0.0)), 6),
        "number_of_components": int(seg_meta.get("number_of_components", 0)),
        "touches_border": bool(seg_meta.get("touches_border", False)),
        "processed_image_path": str(processed_output_path) if processed_output_path is not None else "",
    }
    for i, label in enumerate(LABEL_ORDER):
        row[f"prob_{label}"] = round(float(probs[i]), 6)
    for feature_name, feature_value in raw_feature_dict.items():
        row[f"feature_{feature_name}"] = float(feature_value) if pd.notna(feature_value) else np.nan
    return row


In [6]:
def predict_folder(folder_path, save_csv=True, save_processed_images=True, background_mode=BACKGROUND_MODE):
    source_folder = Path(folder_path)
    image_paths = list_images_recursive(source_folder)
    if len(image_paths) == 0:
        raise ValueError(f"No supported image files found in: {source_folder}")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = INFERENCE_ROOT / f"segmented6_hybrid_best_inference_{stamp}"
    processed_dir = run_dir / "processed_segmented_roi"
    run_dir.mkdir(parents=True, exist_ok=True)
    if save_processed_images:
        processed_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    errors = []

    for image_path in image_paths:
        try:
            relative_path = image_path.relative_to(source_folder)
            processed_output_path = processed_dir / relative_path if save_processed_images else None
            rows.append(
                predict_single_image(
                    image_path=image_path,
                    processed_output_path=processed_output_path,
                    background_mode=background_mode,
                )
            )
        except Exception as exc:
            errors.append({"image_path": str(image_path), "error": repr(exc)})

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values(["freshness_score", "top_confidence"], ascending=[False, False]).reset_index(drop=True)

    summary = {
        "timestamp": stamp,
        "model_path": str(MODEL_PATH),
        "metadata_path": str(METADATA_PATH),
        "source_folder": str(source_folder),
        "run_dir": str(run_dir),
        "processed_dir": str(processed_dir) if save_processed_images else "",
        "background_mode": background_mode,
        "best_fold": BEST_FOLD,
        "best_seed": BEST_SEED,
        "total_images": int(len(image_paths)),
        "successful_predictions": int(len(df)),
        "failed_predictions": int(len(errors)),
        "segmentation_failed_count": int(df["segmentation_failed"].sum()) if len(df) else 0,
        "mean_top_confidence": float(df["top_confidence"].mean()) if len(df) else None,
        "mean_freshness_score": float(df["freshness_score"].mean()) if len(df) else None,
        "class_counts": df["predicted_class"].value_counts().to_dict() if len(df) else {},
        "freshness_band_counts": df["freshness_band"].value_counts().to_dict() if len(df) else {},
    }

    out_csv = None
    errors_csv = None
    summary_json = run_dir / "summary.json"

    if save_csv and len(df) > 0:
        out_csv = run_dir / "predictions.csv"
        df.to_csv(out_csv, index=False)

    if len(errors) > 0:
        errors_csv = run_dir / "errors.csv"
        pd.DataFrame(errors).to_csv(errors_csv, index=False)

    summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return df, errors, summary, out_csv, errors_csv, run_dir


In [ ]:
if IPYWIDGETS_AVAILABLE:
    upload_widget = widgets.FileUpload(
        accept=".zip",
        multiple=False,
        description="Upload ZIP",
    )
    folder_text = widgets.Text(
        value="",
        placeholder="Example: E:/Thesis Code/my_folder_of_images",
        description="Folder path:",
        layout=widgets.Layout(width="700px"),
    )
    run_button = widgets.Button(description="Run Hybrid Inference", button_style="success")
    output_area = widgets.Output()

    display(widgets.HTML("<b>Option A:</b> Upload a ZIP file containing your folder of images."))
    display(upload_widget)
    display(widgets.HTML("<b>Option B:</b> Enter a local folder path and run."))
    display(folder_text)
    display(run_button)
    display(output_area)

    def _run_inference(_):
        with output_area:
            clear_output(wait=True)
            try:
                folder_to_use = None
                uploaded_items = normalize_uploaded_items(upload_widget.value)

                if len(uploaded_items) > 0:
                    print("Using uploaded ZIP file...")
                    folder_to_use = extract_uploaded_zip_to_temp(uploaded_items[0])
                    print("Extracted to:", folder_to_use)
                elif folder_text.value.strip():
                    folder_to_use = Path(folder_text.value.strip())
                    print("Using local folder:", folder_to_use)
                else:
                    print("Please upload a ZIP file or enter a folder path.")
                    return

                df, errors, summary, out_csv, errors_csv, run_dir = predict_folder(
                    folder_to_use,
                    save_csv=True,
                    save_processed_images=True,
                    background_mode=BACKGROUND_MODE,
                )

                print("Summary:")
                print(json.dumps(summary, indent=2))
                print("\nRun directory:", run_dir)
                if out_csv is not None:
                    print("Predictions CSV:", out_csv)
                if errors_csv is not None:
                    print("Errors CSV:", errors_csv)

                print("\nTop predictions:")
                display(df.head(20))

                if len(errors) > 0:
                    print("\nErrors:")
                    display(pd.DataFrame(errors).head(20))

            except Exception as exc:
                print("Error:", exc)

    run_button.on_click(_run_inference)
else:
    print("ipywidgets is not installed in this kernel.")
    print("Use manual mode in the next cell.")

HTML(value='<b>Option A:</b> Upload a ZIP file containing your folder of images.')

FileUpload(value=(), accept='.zip', description='Upload ZIP')

HTML(value='<b>Option B:</b> Enter a local folder path and run.')

Text(value='', description='Folder path:', layout=Layout(width='700px'), placeholder='Example: E:/Thesis Code/…

Button(button_style='success', description='Run Hybrid Inference', style=ButtonStyle())

Output()

In [8]:
# Manual mode
# Set any folder containing source meat images, then run this cell.

MANUAL_FOLDER_PATH = ""

if MANUAL_FOLDER_PATH:
    df, errors, summary, out_csv, errors_csv, run_dir = predict_folder(
        MANUAL_FOLDER_PATH,
        save_csv=True,
        save_processed_images=True,
        background_mode=BACKGROUND_MODE,
    )
    print(json.dumps(summary, indent=2))
    print("Run directory:", run_dir)
    if out_csv is not None:
        print("Predictions CSV:", out_csv)
    if errors_csv is not None:
        print("Errors CSV:", errors_csv)
    display(df.head(30))
    if len(errors) > 0:
        display(pd.DataFrame(errors).head(20))
else:
    print("Set MANUAL_FOLDER_PATH first.")

Set MANUAL_FOLDER_PATH first.
